# Experiment 035: A_GC scientific seed 0 (Colab)

The A_GC pilot passed the memory, precision and reference gates. This session starts the 3B full-parameter seed-0 arm again from update 1, then evaluates the completed endpoint. The failed rung-A attempts and pilot weights are not reused.


## 0. Compute units and Drive


In [1]:
import glob, json, os, time
ALLOCATOR_CONFIG = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ALLOCATOR_CONFIG
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 480.52
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '035_2_colab_agc_science', 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260818_111710 | cu_before = 480.52
Mounted at /content/drive
saved open session: 20260818_111710


## 1. Runtime


In [2]:
import shutil, subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 79, 'need the A100 80GB runtime'
print('local free:', round(shutil.disk_usage('/content').free / 1024**3, 1), 'GiB')
print('drive free:', round(shutil.disk_usage(ROOT).free / 1024**3, 1), 'GiB')


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-2a2730ef-104f-dea4-ed1c-2504a3598346)
total vram: 79.3 GiB
local free: 188.6 GiB
drive free: 179.2 GiB


## 2. Repository and dependencies


In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')


dependencies installed


In [5]:
import getpass, os
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None
print('W&B key loaded')


W&B key loaded


## 3. Wiring checks


In [6]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_035_matched_grpo.py',
    'tests/test_training_wrapper_contracts.py',
    'tests/test_checkpoint.py',
    'tests/test_sentinel.py',
    'tests/test_protocol_freeze.py',
], check=True)
print('wiring checks passed')


wiring checks passed


In [7]:
import json, os, shutil, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s35_matched_grpo import (
    ALLOCATOR_CONFIG, PILOT_RUNGS_035, TRAINING_COMPLETE, arm_tag, bind_start,
    commit_endpoint, commit_execution_lock, commit_pair_parity, commit_start_probe,
    config_path, endpoint_state, prepare_agc_science, safe_close_checkpoints,
)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
TECH = os.path.join(ROOT, 'results', '035_technical')
START_PATH = os.path.join(TECH, 'starts', 'B3', 'seed0.json')
cfg = load_config(config_path('B3', 0, 'full'))
start_record = bind_start(cfg, START_PATH)

selection = read_json(os.path.join(TECH, '035_scale_selection.json'))
override = read_json(os.path.join(TECH, '035_execution_override.json'))
retry = read_json(os.path.join(TECH, '035_allocator_retry.json'))
authorisation = prepare_agc_science(ROOT, selection, override, retry)
pilot = read_json(os.path.join(TECH, 'pilots', 'B3', 'A_GC', 'pilot_report.json'))
assert authorisation['execution'] == pilot['execution'] == PILOT_RUNGS_035['A_GC']
assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == ALLOCATOR_CONFIG
checkpoint_gib = pilot['checkpoint_bytes'] / 1024**3
required_gib = 3 * checkpoint_gib + 50
print(f'science Drive gate: {required_gib:.1f} GiB')
assert shutil.disk_usage(ROOT).free / 1024**3 >= required_gib, 'not enough Drive space'
print('A_GC scientific seed 0 authorised')


science Drive gate: 153.5 GiB
A_GC scientific seed 0 authorised


In [8]:
def append_jsonl(path, row):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def active_training_workers(cfg_path):
    process_list = subprocess.run(
        ['ps', '-eo', 'pid=,args='], capture_output=True, text=True, check=True).stdout
    return [line.strip() for line in process_list.splitlines()
            if 'scripts/s35_worker.py' in line and '--task train' in line
            and f'--config {cfg_path}' in line]

def run_worker(command, status_path, result_dir, label):
    if os.path.exists(status_path):
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    if os.path.exists(status_path):
        status = read_json(status_path)
    else:
        status = {'completed': False, 'error_type': 'MissingStatus',
                  'error_message': 'worker exited without a status file'}
    status = {**status, 'label': label, 'returncode': process.returncode,
              'parent_wall_seconds': round(time.time() - started, 1)}
    append_jsonl(os.path.join(result_dir, '035_attempts.jsonl'), status)
    append_jsonl(os.path.join(ROOT, 'results', 'compute_ledger.jsonl'), {
        'run_tag': label, 'condition': f'035_{status.get("task", "worker")}',
        'timings_sec': {'worker': status['parent_wall_seconds']},
        'train_sec': status['parent_wall_seconds'] if status.get('task') == 'train' else 0,
        'train_steps': status.get('completed_updates', 0),
        'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
    })
    return process, status

def worker_command(task, cfg_path, start_path, rung, status_path, out_dir, result_dir, **kwargs):
    command = [
        sys.executable, 'scripts/s35_worker.py', '--task', task,
        '--config', cfg_path, '--start-record', start_path, '--rung', rung,
        '--status', status_path, '--out-dir', out_dir, '--result-dir', result_dir,
    ]
    for key, value in kwargs.items():
        if value is not None:
            command.extend(['--' + key.replace('_', '-'), str(value)])
    return command


## 4. Scientific arm


In [9]:
def run_start_probe():
    result_dir = os.path.join(ROOT, 'results', '035_start_B3_seed0')
    manifest = os.path.join(result_dir, '035_start_probe_manifest.json')
    if os.path.exists(manifest):
        return commit_start_probe(result_dir, '/content/unused', start_record)
    raise RuntimeError('the committed seed-0 start probe is missing')

def run_parity_pair():
    pair_dir = os.path.join(TECH, 'parity', 'B3', 'seed0', 'A_GC')
    pair_path = os.path.join(pair_dir, 'pair_parity.json')
    os.makedirs(pair_dir, exist_ok=True)
    records = {}
    for mode in ('lora', 'full'):
        record_path = os.path.join(pair_dir, f'{mode}_parity.json')
        if not os.path.exists(record_path):
            local_out = f'/content/035_parity_B3_0_A_GC_{mode}'
            if os.path.isdir(local_out):
                shutil.rmtree(local_out)
            status_path = os.path.join(pair_dir, f'{mode}_status.json')
            command = worker_command(
                'parity', config_path('B3', 0, mode), START_PATH, 'A_GC',
                status_path, local_out, pair_dir, parity_record=record_path)
            process, status = run_worker(
                command, status_path, pair_dir, f'035_parity_B3_seed0_A_GC_{mode}')
            if process.returncode != 0 or not status.get('completed'):
                raise RuntimeError(f'parity failed for B3 seed 0 {mode}')
            if os.path.isdir(local_out):
                shutil.rmtree(local_out)
        records[mode] = read_json(record_path)
    return commit_pair_parity(pair_path, records['lora'], records['full'])


In [10]:
def training_paths():
    tag = arm_tag('B3', 0, 'full')
    return (os.path.join(ROOT, 'results', tag),
            os.path.join(ROOT, 'checkpoints', tag, 'grpo'))

def run_training():
    result_dir, output_dir = training_paths()
    complete_path = os.path.join(result_dir, TRAINING_COMPLETE)
    precision_path = os.path.join(result_dir, 'precision_report.json')
    if os.path.exists(complete_path):
        from scripts.s35_matched_grpo import commit_training_complete
        return commit_training_complete(
            cfg, result_dir, output_dir, start_record, 'A_GC', precision_path)
    cfg_path = config_path('B3', 0, 'full')
    active = active_training_workers(cfg_path)
    assert not active, f'A_GC training worker already active: {active}'
    status_path = os.path.join(result_dir, 'worker_status.json')
    if os.path.exists(status_path):
        previous = read_json(status_path)
        if previous.get('cuda_oom') is True:
            raise RuntimeError('the A_GC scientific attempt already ended with CUDA OOM')
        if previous.get('error_type') not in (None, 'KeyboardInterrupt'):
            raise RuntimeError('the previous A_GC failure requires inspection before resume')
    os.makedirs(result_dir, exist_ok=True)
    command = worker_command(
        'train', cfg_path, START_PATH, 'A_GC', status_path,
        output_dir, result_dir, precision_report=precision_path)
    command.extend(['--report-to', 'tensorboard', 'wandb'])
    process, status = run_worker(command, status_path, result_dir, '035_B3_seed0_full_A_GC')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError('A_GC training failed for B3 seed 0 full')
    return read_json(complete_path)

def run_endpoint(training):
    result_dir, output_dir = training_paths()
    state = endpoint_state(result_dir, training)
    if state == 'tampered':
        raise RuntimeError('committed B3 seed-0 endpoint changed')
    if state == 'committed':
        return read_json(os.path.join(result_dir, '035_eval_manifest.json'))
    scratch = '/content/035_eval_B3_0_full_A_GC'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    status_path = os.path.join(result_dir, 'eval_status.json')
    command = worker_command(
        'eval', config_path('B3', 0, 'full'), START_PATH, 'A_GC', status_path,
        output_dir, result_dir, branch='full', scratch_dir=scratch)
    process, status = run_worker(command, status_path, result_dir, '035_eval_B3_seed0_full_A_GC')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError('A_GC endpoint evaluation failed for B3 seed 0 full')
    record = commit_endpoint(result_dir, scratch, training)
    shutil.rmtree(scratch)
    # remove checkpoints only after the endpoint record commits
    safe_close_checkpoints(result_dir, output_dir)
    return record


In [ ]:
# finish shared start and parity checks before the full arm trains
start_probe = run_start_probe()
parity = run_parity_pair()
training = run_training()
lock = commit_execution_lock(ROOT, 'B3', 'A_GC', training)
endpoint = run_endpoint(training)
print('training complete:', training['completed'])
print('execution lock:', lock['rung'], lock['execution'])
print('endpoint complete:', endpoint['completed'])


Streaming output truncated to the last 5000 lines.
│ │               │ Which of the  │               │              │           │ │
│ │               │ following     │               │              │           │ │
│ │               │ terms is the  │               │              │           │ │
│ │               │ opposite of a │               │              │           │ │
│ │               │ wall          │               │              │           │ │
│ │               │ surface for   │               │              │           │ │
│ │               │ trying soil   │               │              │           │ │
│ │               │ buildings?    │               │              │           │ │
│ │               │ A: east       │               │              │           │ │
│ │               │ B: Sea        │               │              │           │ │
│ │               │ C: him        │               │              │           │ │
│ │               │ D: clear      │               │       

RuntimeError: A_GC training failed for B3 seed 0 full

## 5. Current state


In [ ]:
result_dir, _ = training_paths()
print('authorised rung:', authorisation['rung'])
print('pilot peak reserved GiB:', pilot['peak_reserved_gib'])
print('training:', read_json(os.path.join(result_dir, TRAINING_COMPLETE))['completed'])
print('endpoint:', endpoint_state(
    result_dir, read_json(os.path.join(result_dir, TRAINING_COMPLETE))))


## 6. Close compute units


In [ ]:
cu_after = None
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)
